# 03 — Boundary coverage and facial dependence are different interventions

    This notebook uses one classifier family (boosting) and small samples to keep
    the mechanisms visible. The full scripts fit all six families across three
    independent development runs. These teaching outcomes are not paper results.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from threadpoolctl import threadpool_limits

# The same notebook works from the repository root or notebooks/.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "pyproject.toml").exists() and (p / "src/attendpro_paper2").exists())
thread_limit = threadpool_limits(limits=1)
pd.set_option("display.max_columns", 12)

In [ ]:
from functools import partial
from attendpro_paper2 import evaluate, fit_models, pooled, predict, simulate_attendpro
from attendpro_paper2.modeling import scores, select_threshold
from attendpro_paper2.boundary import boundary
from attendpro_paper2.dependence import CONFIGS, generate

seed = 20264001
train, validation, calibration = pooled(500, seed), pooled(250, seed+100), pooled(250, seed+200)
original = fit_models(train, validation, calibration, seed, ["Gradient boosting"])["Gradient boosting"]

## 1. Add boundary examples while keeping facial evidence

    Half of genuine outsiders and all genuine boundary insiders are moved to the
    near-boundary bands for development. Location observations are regenerated;
    nobody crosses to the other side of the true authorization boundary.

In [ ]:
train_b = boundary(train, seed+700000, fraction=.5)
validation_b = boundary(validation, seed+700001, fraction=.5)
calibration_b = boundary(calibration, seed+700002, fraction=.5)
for before, after in [(train, train_b), (validation, validation_b), (calibration, calibration_b)]:
    for column in ["face_similarity", "liveness_score", "face_quality_score", "truth_valid_presence"]:
        assert np.array_equal(before[column], after[column])

# Only the cutoff changes in this arm; original model and calibrator are kept.
adapted_cutoff = select_threshold(validation_b.truth_valid_presence,
                                   scores(validation_b, original), target=.01)
redeveloped = fit_models(train_b, validation_b, calibration_b, seed,
                          ["Gradient boosting"])["Gradient boosting"]
test = boundary(simulate_attendpro(5000, "nominal", seed+1000), seed+91000, fraction=1)
accepted = {"Frozen original": predict(test, original),
            "Cutoff only": scores(test, original) >= adapted_cutoff,
            "Redeveloped": predict(test, redeveloped)}
rows = []
for arm, decision in accepted.items():
    outside = (test.scenario == "genuine_outside").to_numpy()
    inside = (test.scenario == "genuine_boundary_inside").to_numpy()
    rows.append(dict(arm=arm, **evaluate(test.truth_valid_presence, decision),
                     outsider_acceptance=decision[outside].mean(),
                     boundary_inside_rejection=(~decision[inside]).mean()))
boundary_results = pd.DataFrame(rows)
display(boundary_results.set_index("arm")
        [["FAR", "FRR", "outsider_acceptance", "boundary_inside_rejection"]].mul(100).round(2))

## 2. Change facial dependence without adding location coverage

    With the same seed, copulas share scenario, truth, quality and GPS evidence.
    Their six primitives change jointly; final liveness is recomputed. In particular,
    Student-t with ρ=0 is not independence. A matching copula model uses independent
    development splits drawn under that same condition, not the final test draws.

In [ ]:
dependence_rows = []
preserved = ["scenario", "truth_valid_presence", "gps_accuracy_m", "observed_distance_m",
             "quality_pass", "face_quality_score"]
reference_evidence = simulate_attendpro(2000, "nominal", seed+2000)
for label, family, rho in CONFIGS:
    generator = partial(generate, family=family, rho=rho)
    test_c = generator(2000, "nominal", seed+2000)
    pd.testing.assert_frame_equal(test_c[preserved], reference_evidence[preserved], check_exact=True)
    train_c = pooled(300, seed+10000, generator)
    validation_c = pooled(200, seed+10100, generator)
    calibration_c = pooled(200, seed+10200, generator)
    matched = fit_models(train_c, validation_c, calibration_c, seed,
                           ["Gradient boosting"])["Gradient boosting"]
    for protocol, fitted in [("Frozen", original), ("Matched", matched)]:
        dependence_rows.append(dict(condition=label, protocol=protocol,
                                    **evaluate(test_c.truth_valid_presence, predict(test_c, fitted))))
dependence_results = pd.DataFrame(dependence_rows)
display(dependence_results.pivot(index="condition", columns="protocol", values=["FAR", "FRR"])
        .mul(100).round(2))

In [ ]:
destination = ROOT / "runs/notebook03"
destination.mkdir(parents=True, exist_ok=True)
boundary_results.to_csv(destination / "boundary_teaching_metrics.csv", index=False)
dependence_results.to_csv(destination / "dependence_teaching_metrics.csv", index=False)

### What this notebook does not establish

    One small fit does not establish a stable ranking. Boundary redevelopment changes
    the entire fitting/selection/calibration/cutoff pipeline. Matching facial
    dependence does not supply missing location examples. Neither intervention
    tests malicious GPS reports. Use the archived outputs and full scripts for
    the actual three-fit comparisons.